# 📊 Pipeline de Avaliação — Corretor ENEM

Pipeline completo de benchmark: carrega as redações e gabaritos do `banco.sql`, executa OCR com **Gemini**, corrige com o modelo configurado no `.env`, calcula métricas (MAE por competência) e plota os resultados.

**Configuração necessária:** arquivo `.env` na raiz do projeto com `GEMINI_API_KEY` (obrigatório) e opcionalmente `OPENAI_API_KEY` + `LANGUAGE_MODEL`.

In [ ]:
# Descomente a linha abaixo na primeira execução para instalar as dependências
# %pip install -q python-dotenv google-generativeai pandas matplotlib seaborn openai

import os
import re
import time
import datetime
import pandas as pd
import google.generativeai as genai
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pathlib import Path

try:
    from openai import OpenAI
    _openai_disponivel = True
except ImportError:
    _openai_disponivel = False
    print("⚠️  openai não instalado — correção usará Gemini como fallback.")

print("✅ Imports OK")

In [ ]:
# Carrega variáveis do .env (raiz do projeto, um nível acima de notebooks/)
load_dotenv(dotenv_path=Path("..") / ".env", override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
LANGUAGE_MODEL = os.getenv("LANGUAGE_MODEL", "gpt-4o")

if not GEMINI_API_KEY:
    raise EnvironmentError("❌ GEMINI_API_KEY não encontrada no .env")

genai.configure(api_key=GEMINI_API_KEY)

usar_openai = (
    _openai_disponivel
    and bool(OPENAI_API_KEY)
    and LANGUAGE_MODEL.lower().startswith("gpt")
)

if usar_openai:
    client_openai = OpenAI(api_key=OPENAI_API_KEY)
    print("✅ Gemini (OCR) + OpenAI (Correção) configurados")
else:
    client_openai = None
    LANGUAGE_MODEL = "gemini-2.5-flash"
    print("✅ Gemini configurado para OCR e Correção")

print(f"   Modelo de correção: {LANGUAGE_MODEL}")

In [ ]:
BANCO_SQL = Path("../data/banco.sql")
DATA_DIR  = Path("../data")


def carregar_banco_sql(caminho: Path) -> pd.DataFrame:
    """Parseia os INSERTs do banco.sql e retorna um DataFrame com o gabarito INEP."""
    with open(caminho, encoding="utf-8") as f:
        sql = f.read()
    padrao = r"\(\s*'([^']+)'\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)"
    matches = re.findall(padrao, sql)
    df = pd.DataFrame(
        matches,
        columns=["redacao", "inep_c1", "inep_c2", "inep_c3", "inep_c4", "inep_c5", "inep_total"]
    )
    for col in ["inep_c1", "inep_c2", "inep_c3", "inep_c4", "inep_c5", "inep_total"]:
        df[col] = df[col].astype(int)
    return df


def resolver_caminho_imagem(nome_arquivo: str):
    """Tenta .jpg e .jpeg para localizar o arquivo real na pasta data/."""
    for ext in [".jpg", ".jpeg"]:
        caminho = DATA_DIR / Path(nome_arquivo).with_suffix(ext).name
        if caminho.exists():
            return caminho
    return None


df_banco = carregar_banco_sql(BANCO_SQL)
df_banco["caminho"] = df_banco["redacao"].apply(resolver_caminho_imagem)

encontradas = df_banco["caminho"].notna().sum()
print(f"✅ {len(df_banco)} redações carregadas do banco")
print(f"   Imagens encontradas: {encontradas}/{len(df_banco)}")
df_banco[["redacao", "inep_c1", "inep_c2", "inep_c3", "inep_c4", "inep_c5", "inep_total"]].head()

In [ ]:
def ocr_gemini(caminho_imagem: Path) -> str:
    """Extrai texto manuscrito da imagem usando Gemini 2.5 Flash."""
    model = genai.GenerativeModel("gemini-2.5-flash")
    with open(caminho_imagem, "rb") as f:
        image_bytes = f.read()
    ext = Path(str(caminho_imagem)).suffix.lower()
    mime = "image/jpeg" if ext in (".jpg", ".jpeg") else "image/png"
    response = model.generate_content([
        "Extraia todo o texto manuscrito da imagem com fidelidade máxima. "
        "Retorne apenas o texto extraído, sem comentários ou formatação adicional.",
        {"mime_type": mime, "data": image_bytes},
    ])
    return response.text


print("✅ Função ocr_gemini definida")

In [ ]:
PROMPT_CORRECAO = """<role>
Você é um corretor oficial de redações do ENEM com 15 anos de experiência, profundo conhecimento da Cartilha do Participante do ENEM e dos critérios oficiais do INEP. Sua avaliação é técnica, objetiva e estritamente alinhada ao espelho de correção oficial do INEP. Você não tem medo de atribuir a nota máxima (200) em cada competência se o texto cumprir os requisitos da cartilha. Sua correção é justa e didática.
</role>

<task>
Corrija a redação avaliando cada uma das 5 competências do ENEM. Para cada competência, analise os elementos exigidos, liste as evidências do texto e, SOMENTE COMO CONCLUSÃO da sua análise, atribua uma nota seguindo obrigatoriamente os níveis oficiais (0, 40, 80, 120, 160 ou 200).
</task>

<competencias>
- Competência 1 – Domínio da norma culta: ortografia, acentuação, morfossintaxe, regência, concordância e pontuação.
- Competência 2 – Compreensão da proposta e repertório sociocultural produtivo e legitimamente articulado ao tema.
- Competência 3 – Organização e interpretação de informações: coerência, progressão temática e argumentação.
- Competência 4 – Mecanismos de coesão textual: conectivos, pronomes, conjunções.
- Competência 5 – Proposta de intervenção: agente, ação, modo/meio, efeito e detalhamento.
</competencias>

<constraints>
- As notas devem ser OBRIGATORIAMENTE um dos valores oficiais: 0, 40, 80, 120, 160 ou 200.
- Nunca atribua nota fora desses valores.
- A nota 200 na C1 admite até 2 desvios pontuais menores.
- Se a redação fugir do tema, C2 recebe 0 e as demais no máximo 40.
- Se houver menos de 7 linhas, aplique as penalidades oficiais do INEP.
- Se houver desrespeito aos direitos humanos, C5 recebe 0.
</constraints>

<output_format>
Responda ESTRITAMENTE neste formato Markdown:

## Resultado Final

| Competência | Nota |
|-------------|------|
| C1 – Norma Culta | [nota] |
| C2 – Proposta e Repertório | [nota] |
| C3 – Argumentação | [nota] |
| C4 – Coesão | [nota] |
| C5 – Intervenção | [nota] |
| **TOTAL** | **[soma]** |
</output_format>"""


def corrigir_redacao(texto: str) -> str:
    """Corrige a redação usando o modelo configurado no .env."""
    mensagem = f"<essay>\n{texto}\n</essay>"
    if usar_openai and client_openai:
        resp = client_openai.chat.completions.create(
            model=LANGUAGE_MODEL,
            temperature=0.2,
            messages=[
                {"role": "system", "content": PROMPT_CORRECAO},
                {"role": "user",   "content": mensagem},
            ],
        )
        return resp.choices[0].message.content
    else:
        model = genai.GenerativeModel("gemini-2.5-flash")
        resp = model.generate_content(PROMPT_CORRECAO + "\n\n" + mensagem)
        return resp.text


def extrair_notas(resposta: str) -> dict:
    """Extrai as notas finais da tabela Markdown gerada pelo modelo."""
    notas = {"ia_c1": 0, "ia_c2": 0, "ia_c3": 0, "ia_c4": 0, "ia_c5": 0, "ia_total": 0}
    if not resposta:
        return notas
    padroes = {
        "ia_c1":    r"\|\s*C1.*?\|\s*(\d+)\s*\|",
        "ia_c2":    r"\|\s*C2.*?\|\s*(\d+)\s*\|",
        "ia_c3":    r"\|\s*C3.*?\|\s*(\d+)\s*\|",
        "ia_c4":    r"\|\s*C4.*?\|\s*(\d+)\s*\|",
        "ia_c5":    r"\|\s*C5.*?\|\s*(\d+)\s*\|",
        "ia_total": r"\|\s*\*\*TOTAL\*\*.*?\|\s*\*\*(\d+)\*\*\s*\|",
    }
    for col, pat in padroes.items():
        m = re.search(pat, resposta, re.IGNORECASE)
        if m:
            notas[col] = int(m.group(1))
    return notas


print("✅ Funções corrigir_redacao e extrair_notas definidas")

In [ ]:
# ── Pipeline Principal ────────────────────────────────────────────────────────
resultados = []

for _, row in df_banco.iterrows():
    nome    = row["redacao"]
    caminho = row["caminho"]

    print(f"🔄 {nome} ...", end=" ", flush=True)

    if caminho is None:
        print("❌ imagem não encontrada, pulando.")
        continue

    # 1. OCR
    try:
        texto = ocr_gemini(caminho)
    except Exception as e:
        print(f"❌ OCR: {e}")
        continue

    # 2. Correção
    try:
        resposta = corrigir_redacao(texto)
    except Exception as e:
        print(f"❌ Correção: {e}")
        continue

    # 3. Extrai notas
    notas_ia = extrair_notas(resposta)

    resultados.append({
        "redacao":     nome,
        "inep_c1":     row["inep_c1"],
        "inep_c2":     row["inep_c2"],
        "inep_c3":     row["inep_c3"],
        "inep_c4":     row["inep_c4"],
        "inep_c5":     row["inep_c5"],
        "inep_total":  row["inep_total"],
        "texto_ocr":   texto[:300] + "...",
        "resposta_ia": resposta,
        **notas_ia,
    })

    print(f"✅  INEP={row['inep_total']} | IA={notas_ia['ia_total']}")
    time.sleep(1)  # rate limit

df_resultados = pd.DataFrame(resultados)
print(f"\n📋 Pipeline concluído: {len(df_resultados)}/{len(df_banco)} redações processadas")
df_resultados[["redacao", "inep_total", "ia_total"]].head(15)

In [ ]:
# ── Métricas ──────────────────────────────────────────────────────────────────
for i in range(1, 6):
    df_resultados[f"erro_c{i}"] = abs(df_resultados[f"inep_c{i}"] - df_resultados[f"ia_c{i}"])

df_resultados["erro_total"] = abs(df_resultados["inep_total"] - df_resultados["ia_total"])

mae_total = df_resultados["erro_total"].mean()
mae_comps  = {f"C{i}": df_resultados[f"erro_c{i}"].mean() for i in range(1, 6)}

acertos_exatos = (df_resultados["erro_total"] == 0).sum()
taxa_acerto    = acertos_exatos / len(df_resultados) * 100

print(f"📊 RELATÓRIO DO BENCHMARK")
print(f"Modelo: {LANGUAGE_MODEL}")
print(f"Redações avaliadas: {len(df_resultados)}")
print(f"Acertos exatos (nota total): {acertos_exatos} ({taxa_acerto:.1f}%)")
print(f"\nMAE — Erro Absoluto Médio:")
print(f"  Total: {mae_total:.1f} pts")
for comp, err in mae_comps.items():
    print(f"  {comp}:    {err:.1f} pts")

In [ ]:
# ── Visualizações ─────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f"Benchmark — {LANGUAGE_MODEL}", fontsize=13, fontweight="bold")

# Gráfico 1: Dispersão nota total
sns.scatterplot(data=df_resultados, x="inep_total", y="ia_total", s=100, color="#2563eb", ax=ax1)
lim = max(df_resultados[["inep_total", "ia_total"]].max()) + 50
ax1.plot([0, lim], [0, lim], color="#dc2626", linestyle="--", label="Acerto perfeito")
ax1.set_title("Correlação: INEP vs IA (Nota Total)")
ax1.set_xlabel("Nota INEP"); ax1.set_ylabel("Nota IA")
ax1.set_xlim(0, lim); ax1.set_ylim(0, lim)
ax1.legend(); ax1.grid(True, alpha=0.3)

# Gráfico 2: MAE por competência
comps = list(mae_comps.keys())
erros = list(mae_comps.values())
sns.barplot(x=comps, y=erros, ax=ax2, palette="viridis")
ax2.set_title("Erro Médio por Competência (MAE)")
ax2.set_xlabel("Competência"); ax2.set_ylabel("Erro Médio (pts)")
ax2.grid(axis="y", alpha=0.3)
for i, v in enumerate(erros):
    ax2.text(i, v + 1, f"{v:.1f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("../data/benchmark_plot.png", dpi=150)
plt.show()
print("📈 Gráfico salvo em data/benchmark_plot.png")